# Notebook 3 - Modelado de Consumo Eléctrico con Spark MLlib
Versión final para entrega.

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import pandas as pd
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName("LCL_Modelado")
    .getOrCreate()
)

print("Spark iniciado correctamente")


In [ ]:

# ==========================================
# CONFIGURACIÓN
# ==========================================

MODO_PRUEBA = True

PARQUET_PATH = "../data/processed/lcl_preprocessed_parquet"

print("Modo prueba:", MODO_PRUEBA)
print("Parquet:", PARQUET_PATH)


In [ ]:

# ==========================================
# CARGA DEL DATASET
# ==========================================

df = spark.read.parquet(PARQUET_PATH)

print("Registros originales:")
print(df.count())

df.printSchema()


In [ ]:

# ==========================================
# MODO PRUEBA
# ==========================================

if MODO_PRUEBA:
    df = df.sample(0.01, seed=42)
    print("Dataset reducido para pruebas")
else:
    print("Dataset completo")

print("Registros utilizados:")
print(df.count())


In [ ]:

# ==========================================
# VARIABLES DEL MODELO
# ==========================================

features = [
    "hour",
    "weekday",
    "month",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_48",
    "lag_336",
    "rolling_mean_48",
    "rolling_std_48"
]

target = "target"

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

model_df = assembler.transform(df)

model_df.select("features", "target", "DateTime").show(5, truncate=False)


In [ ]:

# ==========================================
# DIVISIÓN TEMPORAL
# ==========================================

fecha_min = model_df.select(F.min("DateTime")).first()[0]
fecha_max = model_df.select(F.max("DateTime")).first()[0]

print("Fecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)

fecha_corte = "2013-07-01"

train_df = model_df.filter(F.col("DateTime") < fecha_corte)
test_df = model_df.filter(F.col("DateTime") >= fecha_corte)

print("Train:", train_df.count())
print("Test :", test_df.count())


In [ ]:

# ==========================================
# ENTRENAMIENTO
# ==========================================

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="target",
    predictionCol="prediction",
    numTrees=20 if MODO_PRUEBA else 50,
    maxDepth=8 if MODO_PRUEBA else 10,
    seed=42
)

model = rf.fit(train_df)

print("Modelo entrenado")


In [ ]:

# ==========================================
# PREDICCIÓN
# ==========================================

predictions = model.transform(test_df)

predictions.select(
    "target",
    "prediction"
).show(10)


In [ ]:

# ==========================================
# MÉTRICAS
# ==========================================

mae = RegressionEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="mae"
).evaluate(predictions)

rmse = RegressionEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predictions)

r2 = RegressionEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions)

print(f"MAE  : {mae:.5f}")
print(f"RMSE : {rmse:.5f}")
print(f"R²   : {r2:.5f}")


In [ ]:

# ==========================================
# IMPORTANCIA DE VARIABLES
# ==========================================

importances = model.featureImportances.toArray()

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

display(importance_df)


In [ ]:

# ==========================================
# GRÁFICA 1
# IMPORTANCIA DE VARIABLES
# ==========================================

plt.figure(figsize=(9,5))

plt.barh(
    importance_df["Feature"],
    importance_df["Importance"]
)

plt.title("Importancia de variables")
plt.xlabel("Importancia")

plt.tight_layout()
plt.show()


In [ ]:

# ==========================================
# GRÁFICA 2
# REAL VS PREDICHO
# ==========================================

sample = (
    predictions
    .select("target", "prediction")
    .limit(1000)
    .toPandas()
)

plt.figure(figsize=(12,5))

plt.plot(
    sample["target"].values[:200],
    label="Real"
)

plt.plot(
    sample["prediction"].values[:200],
    label="Predicción"
)

plt.title("Consumo real vs predicho")
plt.xlabel("Observación")
plt.ylabel("kWh")

plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

# ==========================================
# GUARDAR MODELO
# ==========================================

MODEL_PATH = "../data/processed/rf_lcl_model"

model.write().overwrite().save(MODEL_PATH)

print("Modelo guardado en:")
print(MODEL_PATH)


In [ ]:

# ==========================================
# RESUMEN AUTOMÁTICO
# ==========================================

print("\n===== RESULTADOS =====")

print(f"MAE  = {mae:.5f}")
print(f"RMSE = {rmse:.5f}")
print(f"R²   = {r2:.5f}")

print("\nInterpretación:")
print("El modelo predice el consumo eléctrico")
print("30 minutos hacia el futuro utilizando")
print("variables temporales y rezagadas.")


In [ ]:

spark.stop()
print("Spark detenido correctamente")
